In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


# ITDA 3rd 학술제 - 소비기한 OCR 추론 노트북
팀: `DScover_카피바라` (itda3-dscover-capybara)

아키텍처 요약은 팀 저장소의 `[DScover_카피바라]_아키텍처구조도.pdf` 및 `README.md` 를 참고하세요.


## 1. 환경 설정
- CPU 전용, 오프라인 실행을 전제로 합니다 (`gpu=False`, `download_enabled=False`).
- 가중치는 `./weights` 폴더에 사전 배치되어 있어야 합니다 (`download_weights.sh` 참고).
- 4-Core vCPU 채점 환경을 가정하여 `ThreadPoolExecutor` 로 이미지 단위 병렬처리를 수행합니다.
  (Jupyter 커널 내부에서 `multiprocessing.Pool(fork)` 를 쓰면, 커널이 이미 여러 백그라운드
  스레드를 띄운 상태이므로 fork 직후 자식 프로세스가 락을 획득하지 못해 멈추는 경우가 있어
  스레드 기반 병렬화로 대체했습니다.)


In [ ]:
import os, sys, time, re, glob, warnings, threading
from datetime import date as _date
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

# 각 스레드에서 torch/BLAS가 내부적으로 다시 멀티스레드를 켜서 서로 경쟁(oversubscription)하는
# 것을 막기 위해, 프로세스 전체의 BLAS 스레드 수를 1로 고정한다. (워커 스레드 수만큼만 코어 활용)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# 일부 numpy/scipy 조합에서는 numpy.testing 모듈이 자체적으로 _blas_supports_fpe 심볼을
# 참조하지만 실제 컴파일된 확장에는 없어 scipy.ndimage import 시점에 크래시가 나는 경우가
# 있다(OCR 기능과 무관한 numpy 내부 테스트 유틸리티 심볼). easyocr이 내부적으로 scipy.ndimage
# 를 import 하므로, 그보다 먼저 더미 값을 채워 넣어 이 문제를 우회한다.
import numpy as _np
try:
    _core_mod = getattr(_np, "_core", None) or getattr(_np, "core", None)
    _umath = getattr(_core_mod, "_multiarray_umath", None) if _core_mod is not None else None
    if _umath is not None and not hasattr(_umath, "_blas_supports_fpe"):
        _umath._blas_supports_fpe = lambda *a, **k: False
except Exception:
    pass

WEIGHTS_DIR = os.path.abspath("./weights")
N_WORKERS = min(4, max(1, os.cpu_count() or 1))   # 채점 환경: Standard 4-Core vCPU

# 명시적 안전 마진 정책: nbconvert 타임아웃(2400s)에서 SAFETY_MARGIN_SEC 만큼 반드시 남기고,
# 그 안에서만 "시간을 더 써서 정확도를 높이는" 최적화를 허용한다. 실제 4-core 환경(Codespaces)
# 실측 결과 모델 로딩·CSV 저장 등 부가 비용은 10초 내외로 작았지만, 채점 서버가 그보다 느릴
# 가능성과 예측 오차에 대비해 250초의 여유를 유지한다. 이 값을 더 줄이는 것은 이번 최적화
# 범위에서 제외한다 (정확도보다 "타임아웃으로 정량 0점"을 피하는 것이 항상 우선).
SAFETY_MARGIN_SEC = 250
TIME_BUDGET_SEC = 2400 - SAFETY_MARGIN_SEC   # 2150

# ── 1·2단계: classical CV 기반 저비용 후보 검출 + 배치 인식 (실제 4-core 환경 실측 근거) ──
# 로컬(2-core) 실측에서는 classical CV 1단계가 2단계 fallback과 비용이 비슷해 폐기했었으나,
# 실제 4-core Codespaces에서 전체 3,352장을 다시 측정해보니 정반대였다: recognize()만 호출하는
# 1단계(후보 16개, 배치 인식)+2단계(실패 시 상위 6개 후보만 대비 보정 재인식) 조합으로 전체를
# 1,475초 만에 다 처리하고 예산(2,250초 기준)의 34%가 남았다 — 이전 우리 방식(9.37%, 54%
# 커버리지)보다 정확도(11.28%)·커버리지(100%) 모두 우수했다. 즉 CRAFT 전체 재검출을 생략하고
# 후보 영역만 인식하는 것 자체가 실제 4-core 환경에서는 훨씬 저렴하다는 뜻이다.
FAST_WORK_WIDTH = 1200
FAST_MAX_BOXES = 16
ROI_RETRY_BOXES = 6

# ── 3단계: 1·2단계가 모두 실패한 이미지에 한해, 남는 시간 예산으로 진짜 전체 재검출 ──
# 1·2단계는 classical CV가 처음부터 후보 박스를 놓치면(예: 각인 위치가 예상 밖) 구조적으로
# 못 찾는다. 이 경우에만, 그리고 시간이 허락할 때만, CLAHE 대비보정 + EasyOCR 전체 파이프라인
# (검출+인식)을 별도 canvas로 1회 더 시도한다. "워커 1개가 이미지 1장에 쓸 수 있는 여유시간"
# = (남은 시간 * N_WORKERS) / 남은 이미지 수 가 클수록 큰 canvas를, 작을수록 작은 canvas를
# 쓰고, 그마저도 안 되면 3단계를 건너뛴다 — 즉 "남는 시간만큼만" 정확도에 재투자하고, 안전
# 마진(SAFETY_MARGIN_SEC) 밑으로는 절대 내려가지 않는다. 임계값은 로컬 2-core 실측(1·2단계와
# 무관하게 별도로 측정한 EasyOCR 전체 파이프라인 처리시간)을 기반으로 하며, 실제 4-core
# 환경에서 재검증을 권장한다.
TIER3_CANVAS_SCHEDULE = [
    (3.6, 550),
    (2.6, 450),
    (1.5, 350),
    (0.8, 250),
]
TIER3_MIN_VIABLE_BUDGET = 0.35   # 이보다도 여유가 없으면 3단계를 생략하고 NONE 유지

print(f"N_WORKERS={N_WORKERS}  WEIGHTS_DIR={WEIGHTS_DIR}")


## 2. 설계 배경 (요약)

상품 뒷면 사진에는 소비기한 외에도 제조일자, 바코드, 품목보고번호, 전화번호, 영양성분,
LOT 번호 등 다양한 숫자가 섞여 있다. 또한 CPU 전용/오프라인/2400초 타임아웃이라는 강한
제약이 있어, 단순히 고해상도로 전체 이미지를 딥러닝 OCR에 넣는 방식은 시간 초과 위험이 크다
(자세한 실험 근거는 아키텍처 요약서 PDF 참고).

이 노트북은 **3단계 구조**를 쓴다 — 앞 두 단계는 값싸게 최대한 많이 맞히고, 3단계는 남는
시간 예산만큼만 정확도에 재투자한다.

1. **1단계 (classical CV 후보 검출 + 배치 인식)**: 형태학적 연산으로 텍스트 줄 후보를 최대
   `FAST_MAX_BOXES`(16)개 찾고, EasyOCR 검출기(CRAFT)는 생략한 채 인식기만 원본 해상도로
   한 번에 배치 호출한다. 처음에는 이 단계를 로컬(2-core) 프로파일링만으로 폐기했었으나(적중률
   8~10%, 처리시간은 2단계와 비슷하다고 측정됨), 실제 4-core Codespaces에서 전체 3,352장을
   다시 측정하니 정반대였다: 1·2단계만으로 전체를 1,475초에 처리하고 정확도(11.28%)도
   당시 우리 방식(9.37%)보다 높았다. 로컬 2-core와 실제 4-core에서 상대적 비용 구조가 다르다는
   뜻이라, 로컬 프로파일링 결과를 실제 채점 환경 검증 없이 최종 결론으로 삼지 않기로 했다.
2. **2단계 (ROI 재시도)**: 1단계가 실패하면, 이미지 전체를 다시 검출하지 않고 1단계가 찾은
   후보 중 상위 `ROI_RETRY_BOXES`(6)개만 CLAHE로 대비를 보정해 재인식한다.
3. **3단계 (전체 재검출, 시간 예산이 허락할 때만)**: 1·2단계 모두 실패하는 경우는 classical CV가
   애초에 후보 박스 자체를 놓친 경우(예: 각인 위치가 예상 밖)라 2단계로는 구조적으로 복구할 수
   없다. 이때 **명시적 안전 마진(`SAFETY_MARGIN_SEC`=250초)을 침범하지 않는 한도 내에서만**
   CLAHE + EasyOCR 전체 파이프라인(검출+인식)을 다시 시도한다. 여유시간(`(남은 시간×N_WORKERS)
   /남은 이미지 수`)이 클수록 큰 canvas를, 작을수록 작은 canvas를 쓰고(`TIER3_CANVAS_SCHEDULE`),
   그마저 안 되면 3단계를 생략해 NONE으로 남긴다. 즉 "시간이 남으면 남을수록 더 정확해지고,
   빠듯해지면 안전 마진을 지키는 선에서 멈추는" 구조다.
4. **키워드 기반 후보 스코어링**: 정규식으로 날짜 형식 후보를 모두 찾은 뒤, 인접 텍스트에
   `소비기한/유통기한/까지` 가 있으면 가점, `제조일자/제조/LOT` 가 있으면 감점하며, 달력 기준
   유효성 검사(`datetime.date`)로 `2026-02-30`처럼 실존하지 않는 날짜는 후보에서 제외한다.
5. **시간 예산 가드레일**: 어느 단계에서든 안전 마진 아래로 내려갈 것으로 판단되면 그 시점
   이후 작업은 즉시 `NONE` 처리하여, 어떤 경우에도 채점 타임아웃(정량 0점)이 발생하지 않도록
   한다.


## 3. 날짜 후보 파서 (정규식 + 키워드 스코어링)

In [ ]:
POS_KEYWORDS = ["소비기한", "유통기한", "품질유지기한", "소비 기한", "유통 기한"]
SUFFIX_KEYWORDS = ["까지", "EXP", "exp"]
NEG_KEYWORDS = ["제조일자", "제조일", "제조년월일", "생산일자", "MFG", "mfg", "제조"]
LOT_KEYWORDS = ["LOT", "Lot", "lot", "로트"]

DATE_RE = re.compile(
    r"(?<!\d)(20\d{2})\s*[.\-/,년]\s*(\d{1,2})\s*[.\-/,월]\s*(\d{1,2})\s*일?(?!\d)"
)
DATE_RE_COMPACT = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")


def _valid_ymd(y, m, d):
    y, m, d = int(y), int(m), int(d)
    if not (2020 <= y <= 2035):
        return False
    if not (1 <= m <= 12):
        return False
    try:
        _date(y, m, d)   # 2월 30일처럼 달력에 없는 날짜(OCR 오인식)를 제거
    except ValueError:
        return False
    return True


def find_date_candidates(text):
    out = []
    for m in DATE_RE.finditer(text):
        y, mo, d = m.group(1), m.group(2), m.group(3)
        if _valid_ymd(y, mo, d):
            out.append((y, mo.zfill(2), d.zfill(2)))
    if not out:
        for m in DATE_RE_COMPACT.finditer(text):
            y, mo, d = m.group(1), m.group(2), m.group(3)
            if _valid_ymd(y, mo, d):
                out.append((y, mo.zfill(2), d.zfill(2)))
    return out


def _cy(box):
    ys = [p[1] for p in box]
    return sum(ys) / len(ys)


def _cx(box):
    xs = [p[0] for p in box]
    return sum(xs) / len(xs)


def _nearby(box_a, box_b, y_thresh_ratio=1.6):
    ya, yb = _cy(box_a), _cy(box_b)
    ha = max(p[1] for p in box_a) - min(p[1] for p in box_a)
    hb = max(p[1] for p in box_b) - min(p[1] for p in box_b)
    h = max(ha, hb, 1)
    return abs(ya - yb) < h * y_thresh_ratio


def stitch_fragments(ocr_results, x_gap_ratio=2.5, y_overlap_ratio=0.6):
    """같은 줄에서 쪼개진 조각들(예: 2022 / 02.14)을 이어붙여 추가 후보 문자열을 만든다."""
    items = list(ocr_results)
    items.sort(key=lambda t: (_cy(t[0]), _cx(t[0])))
    stitched = []
    used = [False] * len(items)
    for i in range(len(items)):
        if used[i]:
            continue
        group = [items[i]]
        used[i] = True
        cy = _cy(items[i][0])
        ch = max(p[1] for p in items[i][0]) - min(p[1] for p in items[i][0])
        last_xmax = max(p[0] for p in items[i][0])
        for j in range(i + 1, len(items)):
            if used[j]:
                continue
            bj = items[j][0]
            cyj = _cy(bj)
            xminj = min(p[0] for p in bj)
            if abs(cyj - cy) < max(ch, 1) * y_overlap_ratio and 0 <= (xminj - last_xmax) < ch * x_gap_ratio:
                group.append(items[j])
                used[j] = True
                last_xmax = max(p[0] for p in bj)
        if len(group) > 1:
            merged_text = " ".join(g[1] for g in group)
            stitched.append((group[0][0], merged_text, min(g[2] for g in group)))
    return stitched


def extract_best_date(ocr_results):
    """ocr_results: [(box[[x,y]x4], text, conf), ...] -> (year, month, day) or (None, None, None)"""
    if not ocr_results:
        return None, None, None
    all_items = list(ocr_results) + stitch_fragments(ocr_results)
    candidates = []
    for (box, text, conf) in all_items:
        dates = find_date_candidates(text)
        if not dates:
            continue
        context_texts = [text]
        for (box2, text2, conf2) in ocr_results:
            if box2 is box:
                continue
            if _nearby(box, box2):
                context_texts.append(text2)
        context = " ".join(context_texts)
        has_pos = any(k in context for k in POS_KEYWORDS)
        has_suffix = any(k in context for k in SUFFIX_KEYWORDS)
        has_neg = any(k in context for k in NEG_KEYWORDS)
        has_lot = any(k in text for k in LOT_KEYWORDS)
        score = 10.0
        if has_pos:
            score += 100
        if has_suffix:
            score += 60
        if has_neg:
            score -= 90
        if has_lot:
            score -= 40
        score += conf * 5
        for (y, mo, d) in dates:
            candidates.append((score, conf, y, mo, d))
    if not candidates:
        return None, None, None
    candidates.sort(key=lambda t: (-t[0], -t[1]))
    _, _, y, mo, d = candidates[0]
    return y, mo, d


## 4. OCR 파이프라인 (1단계 배치 인식 → 2단계 ROI 재시도 → 3단계 전체 재검출)

In [ ]:
import cv2
import numpy as np


def propose_line_boxes(img_bgr, work_width=FAST_WORK_WIDTH, max_width_ratio=0.45,
                        min_width_ratio=0.02, max_boxes=FAST_MAX_BOXES):
    h0, w0 = img_bgr.shape[:2]
    scale = work_width / w0
    small = cv2.resize(img_bgr, (work_width, max(1, int(h0 * scale))))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    grad = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))
    _, bw = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    # 세로 방향으로는 거의 붙이지 않아(커널 높이=1) 서로 다른 줄이 하나로 합쳐지는 것을 방지
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    connected = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    connected = cv2.dilate(connected, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 2)), iterations=1)
    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H, W = small.shape[:2]
    cand = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w < 18 or h < 8:
            continue
        ar = w / float(h)
        if ar < 1.3:
            continue
        if h > H * 0.15:
            continue
        wratio = w / W
        if wratio > max_width_ratio or wratio < min_width_ratio:
            continue   # 문단형 긴 줄(설명문)과 잡음 조각을 제외
        cand.append((w * h, x, y, w, h))
    cand.sort(key=lambda t: -t[0])
    cand = cand[:max_boxes]
    boxes = []
    for area, x, y, w, h in cand:
        boxes.append((x / scale, y / scale, (x + w) / scale, (y + h) / scale))
    return boxes


def _to_recognize_format(boxes):
    return [[int(x0), int(x1), int(y0), int(y1)] for (x0, y0, x1, y1) in boxes]


def run_fast_path(reader, img_bgr, gray):
    """1단계: classical CV 후보 검출 + 배치 인식 (검출기 CRAFT는 생략)."""
    boxes = propose_line_boxes(img_bgr)
    if not boxes:
        return [], []
    hlist = _to_recognize_format(boxes)
    try:
        raw = reader.recognize(gray, horizontal_list=hlist, free_list=[],
                                batch_size=min(FAST_MAX_BOXES, len(boxes)))
        return list(raw), boxes
    except Exception:
        return [], boxes


def run_roi_retry(reader, img_bgr, boxes):
    """2단계: 1단계 후보 중 상위 ROI_RETRY_BOXES개만 대비 보정해 재인식한다
    (이미지 전체를 다시 검출하지 않으므로 저비용)."""
    if not boxes:
        return []
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    hlist = _to_recognize_format(boxes[:ROI_RETRY_BOXES])
    try:
        return list(reader.recognize(enhanced, horizontal_list=hlist, free_list=[],
                                      batch_size=len(hlist), contrast_ths=0.03, adjust_contrast=0.7))
    except Exception:
        return []


def run_full_redetect(reader, img_bgr, canvas):
    """3단계: 1·2단계가 classical CV 박스 자체를 놓쳐 모두 실패했을 때만, 시간이
    허락하는 경우에 이미지 전체를 다시 검출한다 (CLAHE 대비보정 + EasyOCR 전체 파이프라인)."""
    h0, w0 = img_bgr.shape[:2]
    scale = canvas / max(h0, w0)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    small = cv2.resize(enhanced, (max(1, int(w0 * scale)), max(1, int(h0 * scale))))
    small_bgr = cv2.cvtColor(small, cv2.COLOR_GRAY2BGR)
    try:
        raw = reader.readtext(small_bgr, canvas_size=canvas)
    except Exception:
        return []
    results = []
    for (box, text, conf) in raw:
        scaled_box = [[p[0] / scale, p[1] / scale] for p in box]
        results.append((scaled_box, text, conf))
    return results


def process_one_image(reader, path, tier3_canvas):
    """tier3_canvas가 None이면 3단계(전체 재검출)를 생략한다 (시간 예산이 없을 때)."""
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None, None, None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    fast_results, boxes = run_fast_path(reader, img_bgr, gray)
    y, mo, d = extract_best_date(fast_results)
    if y is not None:
        return y, mo, d
    roi_results = run_roi_retry(reader, img_bgr, boxes)
    combined = fast_results + roi_results
    y, mo, d = extract_best_date(combined)
    if y is not None or tier3_canvas is None:
        return y, mo, d
    full_results = run_full_redetect(reader, img_bgr, tier3_canvas)
    y, mo, d = extract_best_date(combined + full_results)
    return y, mo, d


## 5. 병렬 처리 + 3단계 시간 예산 가드레일

`ThreadPoolExecutor` 로 이미지 단위 작업을 워커 스레드에 분배한다. 각 스레드는 최초 호출 시
자신만의 EasyOCR Reader 를 한 번만 로드해 재사용한다(스레드-로컬). 1·2단계는 모든 이미지에
항상 시도한다(저비용). 3단계(전체 재검출)만, 공유 시작 시각과 지금까지 완료된 이미지 수
(스레드 간 락으로 보호되는 공유 카운터)로 **남은 이미지 1장당 남은 "워커-시간"**을 계산해서
— 여유가 크면 큰 canvas로, 작으면 작은 canvas로, 그마저 없으면 3단계 자체를 생략한다
(`TIER3_CANVAS_SCHEDULE`, `TIER3_MIN_VIABLE_BUDGET`). 이 여유시간 계산의 budget은 이미
`SAFETY_MARGIN_SEC`(250초)이 반영된 `TIME_BUDGET_SEC`이므로, 3단계가 아무리 적극적으로 시간을
써도 안전 마진 밑으로는 내려가지 않는다. EasyOCR/torch/OpenCV 의 실제 연산 대부분은 C/C++
구현으로 GIL 을 해제하므로, 스레드 기반으로도 프로세스 기반과 유사한 병렬 처리량을 얻으면서
`fork()` 를 아예 사용하지 않아 Jupyter 커널 환경에서의 안정성을 높인다.


In [ ]:
_thread_local = threading.local()
_progress_lock = threading.Lock()
_progress_state = {"completed": 0}


def _get_worker_reader():
    if not hasattr(_thread_local, "reader"):
        import easyocr
        try:
            import torch
            torch.set_num_threads(1)
        except Exception:
            pass
        _thread_local.reader = easyocr.Reader(
            ["ko", "en"], gpu=False,
            model_storage_directory=WEIGHTS_DIR,
            download_enabled=False,
            verbose=False,
        )
    return _thread_local.reader


def _pick_tier3_canvas(elapsed, completed, total, budget, n_workers):
    """3단계(전체 재검출)에 쓸 canvas를 고른다. n_workers 개 워커가 동시에 처리하므로,
    실제 허용 가능한 이미지당 처리 시간은 (남은 시간 * n_workers) / (남은 이미지 수) 이다
    (이 배수를 빠뜨리면 항상 최소 canvas로만 떨어지는 오류가 생긴다는 것을 확인했다).
    여유가 없으면 None을 반환해 3단계를 생략하고 1·2단계 결과(NONE일 수도 있음)를 그대로 쓴다."""
    remaining_imgs = max(1, total - completed)
    remaining_time = budget - elapsed
    if remaining_time <= 0:
        return None
    per_img_budget = remaining_time * n_workers / remaining_imgs
    for threshold, canvas in TIER3_CANVAS_SCHEDULE:
        if per_img_budget >= threshold:
            return canvas
    if per_img_budget >= TIER3_MIN_VIABLE_BUDGET:
        return TIER3_CANVAS_SCHEDULE[-1][1]
    return None


def _worker_process(path, start_time, budget, total_images, n_workers):
    image_id = os.path.splitext(os.path.basename(path))[0]
    elapsed = time.time() - start_time
    if elapsed > budget:
        # 안전 마진 반영 예산 초과: 1·2단계조차 생략하고 안전하게 NONE 처리
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}
    with _progress_lock:
        completed = _progress_state["completed"]
    tier3_canvas = _pick_tier3_canvas(elapsed, completed, total_images, budget, n_workers)
    try:
        reader = _get_worker_reader()
        y, mo, d = process_one_image(reader, path, tier3_canvas)
    except Exception:
        y, mo, d = None, None, None
    with _progress_lock:
        _progress_state["completed"] += 1
    if y is None:
        return {"image_id": image_id, "year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}
    return {"image_id": image_id, "year": y, "month": mo, "day": d, "final_date": f"{y}-{mo}-{d}"}


## 6. 실행: INPUT_DIR 의 모든 이미지에 대해 추론

In [ ]:
image_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.jpg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.jpeg")) +
    glob.glob(os.path.join(INPUT_DIR, "*.png")) +
    glob.glob(os.path.join(INPUT_DIR, "*.bmp")) +
    glob.glob(os.path.join(INPUT_DIR, "*.webp")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.JPEG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.PNG")) +
    glob.glob(os.path.join(INPUT_DIR, "*.BMP")) +
    glob.glob(os.path.join(INPUT_DIR, "*.WEBP"))
)
print(f"input images: {len(image_files)}")

results = []
if len(image_files) == 0:
    print("경고: INPUT_DIR 에서 이미지를 찾지 못했습니다.")
else:
    start_time = time.time()
    total_images = len(image_files)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(_worker_process, p, start_time, TIME_BUDGET_SEC, total_images, N_WORKERS) for p in image_files]
        for i, fut in enumerate(futures):
            results.append(fut.result())
            if (i + 1) % 200 == 0:
                print(f"  processed {i+1}/{len(image_files)}  elapsed={time.time()-start_time:.1f}s")
    print(f"done. total elapsed={time.time()-start_time:.1f}s")


## 7. submission.csv 저장

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
print(df.head())
print("NONE ratio:", (df["final_date"] == "NONE").mean())
